# 06 — Causal Masking, Dropout, and Multi-Head Attention

**Lecture goal:** close the two remaining gaps between the self-attention from notebook 05 and the real mechanism used inside GPT: stopping tokens from "seeing the future," and running several attention computations in parallel.

## Gap 1: attention can currently see the future

GPT generates text one token at a time, left to right: it predicts token 5 using only tokens 1–4, then predicts token 6 using tokens 1–5, and so on — it never has access to tokens that come *after* the one it's predicting, because during real generation those tokens don't exist yet.

But the self-attention we built in notebook 05 lets *every* token attend to *every* other token — including ones later in the sequence. If we trained with that, the model would learn to "cheat": partially basing its prediction of token 5 on token 7, which is information it will never have at actual generation time. Training would look artificially easy and the model would perform badly in practice.

The fix is called **causal masking** (or "masked self-attention"): force the attention weight from any query token to any *future* key token to be exactly zero.

Let's rebuild our toy example from notebook 05 to demonstrate this concretely.

In [1]:
import torch
import torch.nn as nn

inputs = torch.tensor([
    [0.43, 0.15, 0.89],  # Your
    [0.55, 0.87, 0.66],  # journey
    [0.57, 0.85, 0.64],  # starts
    [0.22, 0.58, 0.33],  # with
    [0.77, 0.25, 0.10],  # one
    [0.05, 0.80, 0.55],  # step
])

torch.manual_seed(789)
d_in, d_out = 3, 2
W_query = nn.Linear(d_in, d_out, bias=False)
W_key = nn.Linear(d_in, d_out, bias=False)
W_value = nn.Linear(d_in, d_out, bias=False)

queries = W_query(inputs)
keys = W_key(inputs)
values = W_value(inputs)

attention_scores = queries @ keys.T
attention_weights = torch.softmax(attention_scores / keys.shape[-1] ** 0.5, dim=-1)
print(attention_weights)

tensor([[0.1921, 0.1646, 0.1652, 0.1550, 0.1721, 0.1510],
        [0.2041, 0.1659, 0.1662, 0.1496, 0.1665, 0.1477],
        [0.2036, 0.1659, 0.1662, 0.1498, 0.1664, 0.1480],
        [0.1869, 0.1667, 0.1668, 0.1571, 0.1661, 0.1564],
        [0.1830, 0.1669, 0.1670, 0.1588, 0.1658, 0.1585],
        [0.1935, 0.1663, 0.1666, 0.1542, 0.1666, 0.1529]],
       grad_fn=<SoftmaxBackward0>)


This is the unmasked attention matrix from notebook 05: row `i`, column `j` is "how much token `i` attends to token `j`." Right now row `0` (`"Your"`) has non-zero weight on columns 1–5 — attending to `"journey"`, `"starts"`, etc., which all come *after* it. We need to zero out every entry above the diagonal.

### Building the mask

`torch.tril` ("triangle, lower") returns a copy of a matrix with everything *above* the main diagonal set to zero, keeping the diagonal and below. Applied to a matrix of all 1s, it gives us exactly the pattern we want: 1 where attention is allowed (current and past tokens), 0 where it should be blocked (future tokens).

In [2]:
context_length = attention_scores.shape[0]
mask_allowed = torch.tril(torch.ones(context_length, context_length))
print(mask_allowed)

tensor([[1., 0., 0., 0., 0., 0.],
        [1., 1., 0., 0., 0., 0.],
        [1., 1., 1., 0., 0., 0.],
        [1., 1., 1., 1., 0., 0.],
        [1., 1., 1., 1., 1., 0.],
        [1., 1., 1., 1., 1., 1.]])


### A naive (but instructive) first attempt: mask after softmax

One way to apply this: run softmax as normal, multiply the result elementwise by the mask (zeroing out the disallowed entries), then divide each row by its new sum so the weights add back up to 1.

In [3]:
masked_weights = attention_weights * mask_allowed
row_sums = masked_weights.sum(dim=-1, keepdim=True)
masked_weights_renormalized = masked_weights / row_sums
print(masked_weights_renormalized)

tensor([[1.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.5517, 0.4483, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.3800, 0.3097, 0.3103, 0.0000, 0.0000, 0.0000],
        [0.2758, 0.2460, 0.2462, 0.2319, 0.0000, 0.0000],
        [0.2175, 0.1983, 0.1984, 0.1888, 0.1971, 0.0000],
        [0.1935, 0.1663, 0.1666, 0.1542, 0.1666, 0.1529]],
       grad_fn=<DivBackward0>)


This works — row 0 now only has weight on itself, row 1 only on tokens 0–1, and so on — but it's wasteful: we compute a full softmax over *all* tokens (including future ones), then throw part of it away and redo the normalization by hand.

### The standard approach: mask the scores with `-inf` *before* softmax

Recall softmax involves `exp(score)`. If we set a score to negative infinity before exponentiating, `exp(-inf) = 0` — that entry contributes exactly nothing to the sum, and softmax handles the renormalization for us automatically, in one pass. We build a mask of the *disallowed* (future) positions this time — the upper triangle, excluding the diagonal — using `torch.triu(..., diagonal=1)`.

In [4]:
mask_disallowed = torch.triu(torch.ones(context_length, context_length), diagonal=1)
print(mask_disallowed)

tensor([[0., 1., 1., 1., 1., 1.],
        [0., 0., 1., 1., 1., 1.],
        [0., 0., 0., 1., 1., 1.],
        [0., 0., 0., 0., 1., 1.],
        [0., 0., 0., 0., 0., 1.],
        [0., 0., 0., 0., 0., 0.]])


In [5]:
masked_scores = attention_scores.masked_fill(mask_disallowed.bool(), -torch.inf)
print(masked_scores)

tensor([[0.2899,   -inf,   -inf,   -inf,   -inf,   -inf],
        [0.4656, 0.1723,   -inf,   -inf,   -inf,   -inf],
        [0.4594, 0.1703, 0.1731,   -inf,   -inf,   -inf],
        [0.2642, 0.1024, 0.1036, 0.0186,   -inf,   -inf],
        [0.2183, 0.0874, 0.0882, 0.0177, 0.0786,   -inf],
        [0.3408, 0.1270, 0.1290, 0.0198, 0.1290, 0.0078]],
       grad_fn=<MaskedFillBackward0>)


In [6]:
attention_weights_causal = torch.softmax(masked_scores / keys.shape[-1] ** 0.5, dim=-1)
print(attention_weights_causal)

print()
print("Matches the renormalize-after approach?",
      torch.allclose(attention_weights_causal, masked_weights_renormalized))

tensor([[1.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.5517, 0.4483, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.3800, 0.3097, 0.3103, 0.0000, 0.0000, 0.0000],
        [0.2758, 0.2460, 0.2462, 0.2319, 0.0000, 0.0000],
        [0.2175, 0.1983, 0.1984, 0.1888, 0.1971, 0.0000],
        [0.1935, 0.1663, 0.1666, 0.1542, 0.1666, 0.1529]],
       grad_fn=<SoftmaxBackward0>)

Matches the renormalize-after approach? True


Same result, computed more directly. This `-inf`-before-softmax trick is the standard way causal masking is implemented in every transformer library, including ours going forward.

## Gap 2 (a smaller one): dropout for regularization

**Dropout** is a general neural-network regularization technique: during training, randomly set some fraction of values to zero (each forward pass, a different random subset), which prevents the network from becoming overly reliant on any single connection and tends to improve generalization to new data. It is only active during training — at inference (actually generating text) it's turned off and every value passes through untouched.

GPT applies dropout to the attention weights, right after the softmax. Let's see it in isolation on a toy tensor first.

In [7]:
torch.manual_seed(123)
dropout = nn.Dropout(p=0.5)  # zero out ~50% of entries, at random, each call

example = torch.ones(6, 6)
print(dropout(example))

tensor([[2., 2., 0., 2., 2., 0.],
        [0., 0., 0., 2., 0., 2.],
        [2., 2., 2., 2., 0., 2.],
        [0., 2., 2., 0., 0., 2.],
        [0., 2., 0., 2., 0., 2.],
        [0., 2., 2., 2., 2., 0.]])


About half the entries became `0`. Notice the *surviving* entries became `2.0`, not `1.0` — this is "inverted dropout": since roughly half the values get zeroed during training, the remaining ones are scaled up by `1 / (1 - p)` (here, `1 / 0.5 = 2`) so that the *expected total* stays the same whether or not dropout happens to be active. This keeps training and inference consistent without needing any extra rescaling at inference time.

## Packaging it all into a `CausalAttention` class

Time to combine causal masking, dropout, and support for a **batch** dimension (so far our toy example has been a single sequence of shape `(num_tokens, d_in)`; real data comes in batches of shape `(batch_size, num_tokens, d_in)`, per notebook 03/04).

One new detail: the mask is not a learnable parameter (it never changes via gradient descent), but we still want it to automatically move to the right device (CPU/GPU) along with the rest of the model, and not count as a trainable weight. PyTorch's `register_buffer` is designed for exactly this: state that belongs to the module but isn't trained.

In [8]:
class CausalAttention(nn.Module):
    def __init__(self, d_in, d_out, context_length, dropout, qkv_bias=False):
        super().__init__()
        self.d_out = d_out
        self.W_query = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.W_key = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.W_value = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.dropout = nn.Dropout(dropout)
        # A buffer: moves with the module (e.g. .to("cuda")) but is not a learnable parameter.
        self.register_buffer(
            "mask", torch.triu(torch.ones(context_length, context_length), diagonal=1)
        )

    def forward(self, x):
        batch_size, num_tokens, d_in = x.shape

        queries = self.W_query(x)
        keys = self.W_key(x)
        values = self.W_value(x)

        # keys.transpose(1, 2) swaps the last two dimensions, per example in the batch:
        # (batch, num_tokens, d_out) -> (batch, d_out, num_tokens), so the matmul below
        # produces (batch, num_tokens, num_tokens) attention scores for every example at once.
        attention_scores = queries @ keys.transpose(1, 2)
        attention_scores.masked_fill_(
            self.mask.bool()[:num_tokens, :num_tokens], -torch.inf
        )

        attention_weights = torch.softmax(attention_scores / keys.shape[-1] ** 0.5, dim=-1)
        attention_weights = self.dropout(attention_weights)

        context_vectors = attention_weights @ values
        return context_vectors

In [9]:
# Simulate a batch of 2 identical sequences by stacking our toy example.
batch = torch.stack([inputs, inputs])
print("Batch shape:", batch.shape)  # (batch_size=2, num_tokens=6, d_in=3)

torch.manual_seed(123)
causal_attention = CausalAttention(d_in=3, d_out=2, context_length=batch.shape[1], dropout=0.0)
context_vectors = causal_attention(batch)
print("Context vectors shape:", context_vectors.shape)

Batch shape: torch.Size([2, 6, 3])
Context vectors shape: torch.Size([2, 6, 2])


`(2, 6, 2)`: 2 examples in the batch, 6 tokens each, 2 numbers per context vector (`d_out`) — causal masking and batching both working correctly.

## Gap 3: multi-head attention

One attention "head" (what we've built so far) learns *one* pattern of relationships between tokens — say, which words modify which nouns. Real language has many simultaneous kinds of relationships (grammar, coreference, topic, tone...). **Multi-head attention** runs several independent attention computations ("heads") in parallel, each with its own learned `W_query`/`W_key`/`W_value`, and combines their outputs — letting different heads specialize in different kinds of relationships.

### The simple (but wasteful) way: a wrapper around several `CausalAttention`s

In [10]:
class MultiHeadAttentionWrapper(nn.Module):
    def __init__(self, d_in, d_out, context_length, dropout, num_heads, qkv_bias=False):
        super().__init__()
        self.heads = nn.ModuleList([
            CausalAttention(d_in, d_out, context_length, dropout, qkv_bias)
            for _ in range(num_heads)
        ])

    def forward(self, x):
        # Run every head independently, then glue their outputs together side by side.
        return torch.cat([head(x) for head in self.heads], dim=-1)

In [11]:
torch.manual_seed(123)
mha_wrapper = MultiHeadAttentionWrapper(d_in=3, d_out=2, context_length=batch.shape[1], dropout=0.0, num_heads=2)
context_vectors = mha_wrapper(batch)
print("Output shape:", context_vectors.shape)  # d_out=2 per head, 2 heads -> 4

Output shape: torch.Size([2, 6, 4])


`(2, 6, 4)`: each of the 2 heads produced a `(2, 6, 2)` output, concatenated along the last dimension into `4`. This works and is easy to understand, but it's computationally wasteful: each head redoes the full attention computation with its own separate matrices in a Python loop, rather than taking advantage of batched matrix operations across heads.

### The efficient way: one matrix, split into heads

Instead of separate weight matrices per head, real implementations use *one* `W_query` (and one `W_key`, one `W_value`) that projects straight to the *full* `d_out` (all heads' worth at once), then **reshape** that output to separate it into `num_heads` chunks. This produces mathematically the same kind of result as the wrapper above, but as a small number of large, efficient matrix operations instead of many small ones in a loop.

The reshaping is the trickiest part conceptually, so let's walk through the shapes:

1. Project: `x @ W_query` → shape `(batch, num_tokens, d_out)`, where `d_out = num_heads * head_dim`.
2. **View** that as `(batch, num_tokens, num_heads, head_dim)` — splitting the last dimension into two, without changing any of the underlying numbers, just how we index them.
3. **Transpose** dimensions 1 and 2 → `(batch, num_heads, num_tokens, head_dim)` — grouping by head so that matrix multiplication (which operates on the last two dimensions) computes attention independently *per head*.
4. Do attention exactly as before (scores, causal mask, softmax, dropout, weighted sum of values) — now every operation implicitly happens once per head, per batch example, in parallel.
5. Transpose back and **view** the heads' outputs side by side again, recovering shape `(batch, num_tokens, d_out)`.
6. One final learned linear layer (`out_proj`) mixes information across heads before passing the result onward — without it, the heads' outputs would just sit side-by-side with no interaction.

In [12]:
class MultiHeadAttention(nn.Module):
    def __init__(self, d_in, d_out, context_length, dropout, num_heads, qkv_bias=False):
        super().__init__()
        assert d_out % num_heads == 0, "d_out must be divisible by num_heads"

        self.d_out = d_out
        self.num_heads = num_heads
        self.head_dim = d_out // num_heads

        self.W_query = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.W_key = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.W_value = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.out_proj = nn.Linear(d_out, d_out)  # mixes information across heads
        self.dropout = nn.Dropout(dropout)
        self.register_buffer(
            "mask", torch.triu(torch.ones(context_length, context_length), diagonal=1)
        )

    def forward(self, x):
        batch_size, num_tokens, d_in = x.shape

        queries = self.W_query(x)  # (batch, num_tokens, d_out)
        keys = self.W_key(x)
        values = self.W_value(x)

        # Split d_out into (num_heads, head_dim), then group by head.
        queries = queries.view(batch_size, num_tokens, self.num_heads, self.head_dim).transpose(1, 2)
        keys = keys.view(batch_size, num_tokens, self.num_heads, self.head_dim).transpose(1, 2)
        values = values.view(batch_size, num_tokens, self.num_heads, self.head_dim).transpose(1, 2)
        # All four now have shape (batch, num_heads, num_tokens, head_dim)

        attention_scores = queries @ keys.transpose(2, 3)  # (batch, num_heads, num_tokens, num_tokens)
        attention_scores.masked_fill_(
            self.mask.bool()[:num_tokens, :num_tokens], -torch.inf
        )
        attention_weights = torch.softmax(attention_scores / self.head_dim ** 0.5, dim=-1)
        attention_weights = self.dropout(attention_weights)

        context_vectors = attention_weights @ values  # (batch, num_heads, num_tokens, head_dim)
        context_vectors = context_vectors.transpose(1, 2)  # (batch, num_tokens, num_heads, head_dim)
        # .contiguous() ensures the tensor is laid out in memory correctly after transposing,
        # which .view() requires.
        context_vectors = context_vectors.contiguous().view(batch_size, num_tokens, self.d_out)

        return self.out_proj(context_vectors)

In [13]:
torch.manual_seed(123)
mha = MultiHeadAttention(d_in=3, d_out=4, context_length=batch.shape[1], dropout=0.0, num_heads=2)
context_vectors = mha(batch)
print("Output shape:", context_vectors.shape)

Output shape: torch.Size([2, 6, 4])


Same `(2, 6, 4)` output shape as the wrapper version, but computed with far fewer, larger operations — this is the `MultiHeadAttention` implementation we'll use inside the actual GPT model starting in notebook 08.

## Recap

- **Causal masking** sets future-token attention scores to `-inf` before softmax, so a query can never attend to tokens later in the sequence — essential so training matches how the model actually generates text (left to right, one token at a time).
- **Dropout** randomly zeroes attention weights during training (never at inference) to reduce overfitting.
- **Multi-head attention** runs several attention computations in parallel, each able to specialize in a different kind of token relationship, then combines and mixes their outputs with a final linear projection.

### What's next

We now have a complete, efficient multi-head causal self-attention mechanism. But attention is only one half of a transformer block — in notebook 07 we'll add the other half: layer normalization, a feed-forward network, and residual ("shortcut") connections, and assemble them together into a full `TransformerBlock`.